 links:
 
 https://reflect.run/articles/how-to-deal-with-staleelementreferenceexception-in-selenium/


In [115]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException, StaleElementReferenceException, ElementClickInterceptedException
import os
import time
import requests
from bs4 import BeautifulSoup

In [116]:
base_url = "https://scholarworks.indianapolis.iu.edu"

In [132]:
# output_dir = "html_extracted_Christina"
# output_dir = "html_extracted_Krista"
# output_dir = "html_extracted_Richard"
output_dir = "html_extracted_Silvia"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [124]:
## Approach
'''
try
Downloading the file
sending requests
fetching the response type and writing it
exception handling
error
'''
def download_html(page_url, file_name):
    try:
        response = requests.get(page_url, timeout=10)
        response.raise_for_status()

      
        file_path = os.path.join(output_dir, f"{file_name}.html")
        with open(file_path, "w", encoding="utf-8") as output_file:
            output_file.write(response.text)
        print(f"Saved: {file_path}")
        return file_path
    except requests.exceptions.RequestException as e:
        print(f"Error downloading HTML: {page_url}, Error: {e}")
        return None


In [125]:
# Approach
"""
Setting a soup to drive.page_source html parser

intializing pdf_links = set() 
logic 
anchor tags with href

checking substring item in href
to look for pdfs we base_link + full_url

calling function(url extracted, file_name)
"""
def extract_html(driver):
    pdf_links = set()
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "a")))
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        for anchor in soup.find_all("a", href=True):
            href = anchor['href']
            if "items/" in href:
                full_url = base_url + href if href.startswith("/") else href
                print(f"Downloading: {full_url}")

                file_name = href.split("/")[-1]
                download_html(full_url, file_name)
    except TimeoutException:
        print("Timeout --> Extract _html")
    return pdf_links

In [126]:
### helper function
def process_page(driver):
    try:
        pdf_links = extract_html(driver) # func call
        for pdf_link, file_name in pdf_links:
            download_html(pdf_link, file_name) # func call

    except Exception as e:
        print(f"Error processing page: {e}")

In [127]:
## approach for pagination and html page extraction
'''
Waiting for the button to be clickable
expected condition button to be clickable by CSS selector
Syntax --> a[aria-label='Next']

checking if the button is disabled 

if disabled button then print next button is disabled
page reload time.sleep(5)

exception handling
StaleElementReference 
Exception during navigation
'''

def navigation(starting_url):
    try:
        driver = webdriver.Chrome()  
        driver.get(starting_url)

        while True:
            print("Processing page..")
            process_page(driver)
            try:
                next_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "li.page-item.ng-star-inserted > a[aria-label='Next']")) # next label CSS part
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", next_button) ###
                time.sleep(2) 
                if not next_button.is_displayed() or "disabled" in next_button.get_attribute("class"):
                    print("Next button is disabled or not clickable. Exiting the loop.")
                    break
                
                print("Clicking the next button...")
                next_button.click()
                time.sleep(3)
                
            except StaleElementReferenceException:
                print("Stale element reference detected. Retrying to find the next button...")
                continue
            except ElementClickInterceptedException:
                print("Element click intercepted. Scrolling and retrying...")
                driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
                time.sleep(1)
                next_button.click()
                time.sleep(2)
            except Exception as e:
                print(f"Unexpected error during navigation: {e}")
                break

    except WebDriverException as e:
        print(f"WebDriver error occurred: {e}")
    finally:
        if 'driver' in locals():
            driver.quit()

In [133]:

if __name__ == "__main__":
    # starting_url = "https://scholarworks.indianapolis.iu.edu/collections/e550acea-5875-472c-9edd-4f3db4583e66?cp.page=1" 
    # starting_url = "https://scholarworks.indianapolis.iu.edu/collections/89a8bd48-a912-4b6d-89e6-b880d146914d"
    # starting_url = "https://scholarworks.indianapolis.iu.edu/collections/2a932fe8-5b2f-4d28-a48f-9f17af319ee5"
    # starting_url = "https://scholarworks.indianapolis.iu.edu/collections/c8cb363d-baf2-4bcb-b8e3-08fe4cf0d295"
    starting_url = "https://scholarworks.indianapolis.iu.edu/collections/0af3886d-b9d5-42eb-8788-2f0201eeb26b"
    navigation(starting_url)


Processing page..
Downloading: https://scholarworks.indianapolis.iu.edu/items/3220a090-7b50-4efa-93f2-7e9998af3ccc
Saved: html_extracted_Silvia\3220a090-7b50-4efa-93f2-7e9998af3ccc.html
Downloading: https://scholarworks.indianapolis.iu.edu/items/3220a090-7b50-4efa-93f2-7e9998af3ccc
Saved: html_extracted_Silvia\3220a090-7b50-4efa-93f2-7e9998af3ccc.html
Downloading: https://scholarworks.indianapolis.iu.edu/items/290422c0-9a63-4fd0-846a-dc952da895bf
Saved: html_extracted_Silvia\290422c0-9a63-4fd0-846a-dc952da895bf.html
Downloading: https://scholarworks.indianapolis.iu.edu/items/290422c0-9a63-4fd0-846a-dc952da895bf
Saved: html_extracted_Silvia\290422c0-9a63-4fd0-846a-dc952da895bf.html
Downloading: https://scholarworks.indianapolis.iu.edu/items/69800199-884f-4128-95ae-f14b89336cfa
Saved: html_extracted_Silvia\69800199-884f-4128-95ae-f14b89336cfa.html
Downloading: https://scholarworks.indianapolis.iu.edu/items/69800199-884f-4128-95ae-f14b89336cfa
Saved: html_extracted_Silvia\69800199-884f-41